In [ ]:
# NOTE: This data-generation notebook is co-authored and maintained by Claude.
# Per the repo convention the setup / seed notebooks are Claude-owned (user reviews, refines structure,
# and optimizes); the pipeline and transformation code is written by the user.

from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
import random

In [0]:
# 🏗️ Setup: Shadowfax Logistics Data Platform## Practice Project — Data Generation**Scenario:** Shadowfax Logistics is a mid-size logistics company based in Edoras. They operate a fleet of 200+ delivery vehicles across Middle-earth, handle ~50K shipments/month, and serve both B2B and B2C customers.**Your role:** You're building their data platform on Databricks. The company has multiple data sources:- **Shipment events** — GPS tracking pings, status updates (picked up, in transit, delivered, failed)- **Customer/partner data** — company profiles that change over time (addresses, tiers, contacts)- **Orders** — order lifecycle with payments, cancellations, returns- **Vehicle telemetry** — sensor data from delivery vehicles (speed, fuel, temperature for cold-chain)- **Vehicles** — fleet master (type, capacity, cold-chain capability, home depot); slow-moving SCD1 reference**Run this notebook first** to generate all source data. Then work through notebooks 01-08 in order.---### Setup Instructions1. Import all notebooks into a Databricks workspace (Free Edition works)2. Run this notebook to generate source data3. Work through each notebook — they build on each other

In [0]:
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
print(configs)

In [0]:
# ── Configs ──
ENV = configs.get('env', 'dev')

CATALOG = f"sl_{ENV}"
SCHEMA_INGEST = ENV
SCHEMA_BRONZE = "bronze"  
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
LANDING_VOL = "landing"
LANDING_BASE = f"/Volumes/sl_ingest/{SCHEMA_INGEST}/{LANDING_VOL}"

In [0]:
# Clean up workspace
for catalog in [CATALOG, "sl_ingest"]:
    spark.sql(f"DROP CATALOG IF EXISTS {catalog} CASCADE")
    
display(spark.sql("SHOW CATALOGS"))

In [0]:
# Create catalogs
for catalog in [CATALOG, "sl_ingest"]:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    
display(spark.sql("SHOW CATALOGS"))

In [0]:


# Create schemas in env catalog (use default catalog on Community Edition)
for schema in [SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{schema}.checkpoints")
    
display(spark.sql("SHOW SCHEMAS IN workspace"))


In [0]:
#the rest of this notebook uses the ingest catalog
spark.sql("USE CATALOG sl_ingest")

In [0]:
#create schema and volume in sl_ingest catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_INGEST}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA_INGEST}.{LANDING_VOL}")

## Generate Source Data

In [0]:
random.seed(42)
BASE_DATE = datetime(2025, 1, 1)

# ── CUSTOMERS (200 companies) ──
# Simulates the operational Postgres "customers" table: CURRENT STATE ONLY,
# one row per customer. No SCD2 bookkeeping lives here (no is_current, no version
# history). signup_date = created_at, last_updated = last modification time.
# The warehouse builds the Type 2 dimension downstream from this table's CDC feed.
me_cities = ["Minas Tirith", "Osgiliath", "Pelargir", "Dol Amroth", "Edoras",
             "Helm's Deep", "Aldburg", "Hobbiton", "Bywater", "Michel Delving",
             "Bree", "Rivendell", "Grey Havens", "Tharbad", "Esgaroth", "Dale",
             "Erebor", "Caras Galadhon"]
tiers = ["standard", "premium", "enterprise"]
industries = ["pharma", "retail", "manufacturing", "food_beverage", "tech", "finance"]

# Customer names: canonical Middle-earth characters first (The Lord of the Rings,
# The Hobbit, The Silmarillion), then generated Shire-style names once canon runs out.
me_characters = [
    # Hobbits of the Shire (and thereabouts)
    "Frodo Baggins", "Bilbo Baggins", "Samwise Gamgee", "Meriadoc Brandybuck",
    "Peregrin Took", "Rosie Cotton", "Hamfast Gamgee", "Lobelia Sackville-Baggins",
    "Otho Sackville-Baggins", "Lotho Sackville-Baggins", "Fredegar Bolger",
    "Folco Boffin", "Drogo Baggins", "Primula Brandybuck", "Paladin Took",
    "Eglantine Banks", "Esmeralda Brandybuck", "Saradoc Brandybuck", "Gerontius Took",
    "Bandobras Took", "Belladonna Took", "Bungo Baggins", "Elanor Gamgee",
    "Farmer Maggot", "Ted Sandyman", "Will Whitfoot", "Robin Smallburrow",
    "Tolman Cotton", "Holman Greenhand", "Halfast Gamgee", "Angelica Baggins",
    "Milo Burrows", "Dora Baggins", "Adelard Took", "Everard Took",
    "Melilot Brandybuck", "Celandine Brandybuck", "Mentha Brandybuck",
    "Pervinca Took", "Pearl Took", "Pimpernel Took", "Diamond of Long Cleeve",
    "Estella Bolger", "Déagol", "Sméagol", "Marcho Fallohide", "Blanco Fallohide",
    "Bucca of the Marish",
    # Bree-land and stranger folk
    "Barliman Butterbur", "Bill Ferny", "Harry Goatleaf", "Tom Bombadil", "Goldberry",
    # Men of Gondor, Rohan, Dale and the North
    "Aragorn Elessar", "Boromir", "Faramir", "Denethor II", "Théoden", "Éomer",
    "Éowyn", "Gríma Wormtongue", "Bard the Bowman", "Beorn", "Théodred",
    "Erkenbrand", "Gamling", "Háma", "Elfhelm", "Grimbold", "Dúnhere",
    "Imrahil of Dol Amroth", "Forlong the Fat", "Húrin of the Keys", "Beregond",
    "Bergil", "Ioreth", "Ingold", "Damrod", "Mablung of Ithilien", "Anborn",
    "Halbarad", "Arathorn", "Ecthelion II", "Isildur", "Anárion", "Elendil",
    "Meneldil", "Cirion", "Eorl the Young", "Helm Hammerhand", "Fréaláf Hildeson",
    "Brytta Léofa", "Walda", "Folca", "Fengel", "Thengel", "Morwen of Lossarnach",
    "Girion of Dale", "Brand of Dale", "Bain of Dale", "Éothain", "Céorl",
    "Targon", "Golasgil", "Hirluin the Fair", "Duinhir", "Dervorin", "Angbor",
    # Elves
    "Legolas", "Galadriel", "Celeborn", "Elrond", "Arwen Undómiel", "Glorfindel",
    "Erestor", "Lindir", "Haldir", "Rúmil", "Orophin", "Thranduil", "Círdan",
    "Gildor Inglorion", "Celebrían", "Elladan", "Elrohir", "Fëanor", "Fingolfin",
    "Finarfin", "Fingon", "Turgon", "Finrod Felagund", "Angrod", "Aegnor",
    "Maedhros", "Maglor", "Celegorm", "Caranthir", "Curufin", "Amrod", "Amras",
    "Celebrimbor", "Gil-galad", "Elu Thingol", "Lúthien Tinúviel", "Beleg Cúthalion",
    "Mablung of Doriath", "Daeron", "Eöl", "Maeglin", "Aredhel", "Idril Celebrindal",
    "Voronwë", "Ecthelion of the Fountain", "Egalmoth", "Duilin", "Galdor",
    "Nimrodel", "Amroth", "Finduilas", "Orodreth", "Gwindor", "Saeros", "Tauriel",
    # Dwarves
    "Thorin Oakenshield", "Balin", "Dwalin", "Fíli", "Kíli", "Dori", "Nori", "Ori",
    "Óin", "Glóin", "Bifur", "Bofur", "Bombur", "Gimli", "Dáin Ironfoot",
    "Thráin II", "Thrór", "Durin the Deathless", "Telchar", "Mîm", "Azaghâl",
    "Náin", "Frerin", "Dís", "Fundin", "Gróin", "Borin", "Farin",
    # Wizards, Ainur and other powers
    "Gandalf the Grey", "Saruman the White", "Radagast the Brown", "Alatar",
    "Pallando", "Melian", "Ossë", "Uinen", "Eönwë", "Ilmarë", "Arien", "Tilion",
    "Manwë", "Varda Elentári", "Ulmo", "Aulë", "Yavanna", "Oromë", "Vána",
    "Námo Mandos", "Vairë", "Irmo Lórien", "Estë", "Nienna", "Tulkas", "Nessa",
    # Men of the First Age
    "Beren Erchamion", "Barahir", "Húrin Thalion", "Huor", "Morwen Eledhwen",
    "Rían", "Túrin Turambar", "Niënor Níniel", "Tuor", "Eärendil", "Elwing",
    "Hador Lórindol", "Bëor the Old", "Haleth", "Brandir", "Dorlas", "Emeldir",
    "Aerin", "Ulfang", "Bór", "Amlach", "Marach",
    # Ents and other notable clients
    "Treebeard", "Quickbeam", "Leaflock", "Skinbark", "Ghân-buri-Ghân", "Smaug",
]
me_first_names = ["Tolman", "Wilcome", "Hobson", "Andwise", "Anson", "Hamson",
                  "Halfred", "Daisy", "May", "Marigold", "Hilda", "Mirabella",
                  "Donnamira", "Isengrim", "Isumbras", "Ferumbras", "Fortinbras",
                  "Flambard", "Sigismond", "Hildigrim", "Adalgrim", "Reginard",
                  "Odovacar", "Rudigar", "Herugar", "Madoc", "Marmadoc", "Gorbadoc",
                  "Rorimac", "Dinodas", "Marmadas", "Seredic", "Léofwine", "Éadric",
                  "Fréawine", "Goldwine", "Déor", "Gram", "Baldor", "Aldor"]
me_surnames = ["Bracegirdle", "Proudfoot", "Hornblower", "Goodbody", "Brockhouse",
               "Chubb", "Grubb", "Burrows", "Sandheaver", "Tunnelly", "Underhill",
               "Greenhand", "Whitfoot", "Noakes", "Rumble", "Twofoot", "Longholes",
               "Banks", "Brownlock", "Puddifoot", "Oldbuck", "Boffin", "Bolger",
               "Goodchild", "Cotton", "Gamwich", "Roper", "Gardner", "of Bree",
               "of Dale", "of Esgaroth", "of the Mark", "of Lossarnach"]

def character_for(cid):
    if cid <= len(me_characters):
        return me_characters[cid - 1]
    return f"{random.choice(me_first_names)} {random.choice(me_surnames)}"

def email_for(name, cid):
    import unicodedata
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    slug = "".join(ch if ch.isalnum() else "." for ch in ascii_name.lower().replace("'", ""))
    slug = ".".join(p for p in slug.split(".") if p)
    return f"{slug}.{cid}@ravenpost.me"


customer_records = []
for cid in range(1, 201):
    name = character_for(cid)
    city = random.choice(me_cities)
    tier = random.choices(tiers, weights=[50, 35, 15])[0]
    industry = random.choice(industries)
    signup = BASE_DATE - timedelta(days=random.randint(60, 1500))
    # Most rows were last touched at signup; ~30% had an in-system edit since then.
    last_updated = signup + timedelta(days=random.randint(30, 365)) if random.random() < 0.3 else signup
    customer_records.append((
        cid, name, email_for(name, cid),
        f"{random.randint(1,200)} {random.choice(['Bagshot Row','Great East Road','The Greenway','Old South Road'])}",
        city, f"{random.randint(1000,9999)}", tier, industry, signup, last_updated
    ))

cust_schema = StructType([
    StructField("customer_id", IntegerType()), StructField("company_name", StringType()),
    StructField("contact_email", StringType()), StructField("address", StringType()),
    StructField("city", StringType()), StructField("postal_code", StringType()),
    StructField("tier", StringType()), StructField("industry", StringType()),
    StructField("signup_date", TimestampType()), StructField("last_updated", TimestampType())
])

df_customers = spark.createDataFrame(customer_records, cust_schema)
df_customers.write.mode("overwrite").option("delta.enableChangeDataFeed", "true").saveAsTable(f"{SCHEMA_INGEST}.customers")
print(f"✓ customers: {df_customers.count()} current-state rows (one per customer, Postgres-style)")

In [ ]:
# ── ORDERS (50K orders over 12 months) — file-based CDC SNAPSHOT ──
# Orders are delivered as a file-based CDC feed on the landing volume, NOT as a Delta
# table. This cell writes the INITIAL SNAPSHOT: one full row per order, tagged
# _change_type = "SNAPSHOT" (the initial-load marker, analogous to Debezium's "r"/read op).
# The incremental generator later appends INSERT / UPDATE change rows to the same path.
# _change_ts is the sequence column the SCD2 build orders versions by; the snapshot uses
# each order's order_date, so every later change (stamped at batch time) sorts after it.
# Spark's JSON writer drops null fields per row, so later sparse UPDATE rows will carry
# only their key + changed columns, exactly like a real change feed.
products = [
    ("PKG_STD", "Standard Parcel", 12.50), ("PKG_EXP", "Express Parcel", 24.90),
    ("PKG_SAME", "Same-Day Delivery", 39.90), ("FRT_PAL", "Pallet Freight", 89.00),
    ("FRT_FTL", "Full Truck Load", 450.00), ("COLD", "Cold Chain Parcel", 34.90),
    ("DOC", "Document Courier", 8.90), ("INTL", "International Shipment", 65.00),
]
statuses = ["created", "confirmed", "picked_up", "in_transit", "out_for_delivery",
            "delivered", "failed_delivery", "returned"]
payment_methods = ["invoice", "credit_card", "bank_transfer", "paypal"]

order_records = []
for oid in range(1, 50001):
    cid = random.randint(1, 200)
    prod = random.choice(products)
    qty = random.randint(1, 10) if prod[0].startswith("PKG") else random.randint(1, 3)
    order_date = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(6, 22),
                                        minutes=random.randint(0, 59))
    status_idx = random.choices(range(len(statuses)), weights=[5,5,8,15,10,45,7,5])[0]
    status = statuses[status_idx]
    delivery_date = order_date + timedelta(days=random.randint(1, 5)) if status in ("delivered", "returned") else None
    amount = __builtins__.round(prod[2] * qty * random.uniform(0.9, 1.1), 2)
    if random.random() < 0.02:
        amount = None  # ~2% missing amounts (data-quality wrinkle for the silver layer)
    order_records.append((
        f"ORD-{oid:06d}", cid, prod[0], prod[1], qty, amount,
        status, random.choice(payment_methods), order_date, delivery_date,
        random.choice(me_cities), random.choice(me_cities),
        "SNAPSHOT", order_date,
    ))

order_schema = StructType([
    StructField("order_id", StringType()), StructField("customer_id", IntegerType()),
    StructField("product_code", StringType()), StructField("product_name", StringType()),
    StructField("quantity", IntegerType()), StructField("total_amount", DoubleType(), True),
    StructField("status", StringType()), StructField("payment_method", StringType()),
    StructField("order_date", TimestampType()), StructField("delivery_date", TimestampType(), True),
    StructField("origin_city", StringType()), StructField("destination_city", StringType()),
    StructField("_change_type", StringType()), StructField("_change_ts", TimestampType()),
])

df_orders = spark.createDataFrame(order_records, order_schema)
(df_orders.repartition(10)   # ~5K rows/file
   .write.mode("overwrite")  # initial snapshot seed
   .json(f"{LANDING_BASE}/orders"))
print(f"✓ orders snapshot: {df_orders.count():,} rows -> {LANDING_BASE}/orders (file CDC, _change_type=SNAPSHOT)")


In [0]:
# ── SHIPMENT EVENTS (GPS tracking + status updates, ~500K events) ──
# This is the high-volume streaming-like data

event_types = ["gps_ping", "status_change", "temperature_alert", "delay_alert", "signature_captured"]

shipment_events = []
for i in range(500000):
    oid = f"ORD-{random.randint(1, 50000):06d}"
    vehicle = f"VH-{random.randint(1, 200):03d}"
    evt_type = random.choices(event_types, weights=[60, 20, 5, 10, 5])[0]
    ts = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(0, 23),
                                minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    
    # GPS coordinates (fictional Middle-earth grid; numeric ranges kept from the original seed)
    lat = __builtins__.round(random.uniform(46.0, 47.8), 6) if evt_type == "gps_ping" else None
    lon = __builtins__.round(random.uniform(6.0, 10.5), 6) if evt_type == "gps_ping" else None
    
    # Status for status_change events
    status = random.choice(statuses) if evt_type == "status_change" else None
    
    # Temperature for cold chain alerts
    temp = __builtins__.round(random.gauss(4, 2), 1) if evt_type == "temperature_alert" else None
    
    # Inject duplicates (~3% — common in real event streams)
    shipment_events.append((
        f"EVT-{i+1:07d}", oid, vehicle, evt_type, ts, lat, lon, status, temp,
        random.choice(["kafka", "iot_hub", "api"]),  # source system
    ))
    if random.random() < 0.03:
        shipment_events.append((
            f"EVT-{i+1:07d}", oid, vehicle, evt_type, ts, lat, lon, status, temp,
            random.choice(["kafka", "iot_hub", "api"]),
        ))

event_schema = StructType([
    StructField("event_id", StringType()), StructField("order_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("event_type", StringType()),
    StructField("event_timestamp", TimestampType()),
    StructField("latitude", DoubleType(), True), StructField("longitude", DoubleType(), True),
    StructField("shipment_status", StringType(), True),
    StructField("temperature_celsius", DoubleType(), True),
    StructField("source_system", StringType()),
])

df_events = spark.createDataFrame(shipment_events, event_schema)
(df_events.repartition(100)  # ~5K/file, ~1-2 MB
   .write.mode("overwrite")  # initial seed only
   .json(f"{LANDING_BASE}/shipment_events"))

In [0]:
df_events_validate = spark.read.json(f"{LANDING_BASE}/shipment_events/")
print(f"✓ shipment_events: {df_events_validate.count()} records (includes ~3% intentional duplicates)")

In [0]:
import uuid
# ── VEHICLE TELEMETRY (sensor readings, ~200K rows, SKEWED by vehicle) ──
# vehicle VH-001 to VH-005 generate 60% of all data (skew for optimization practice)

telemetry = []
for i in range(200000):
    # Intentional skew: 60% of readings from 5 vehicles
    if random.random() < 0.6:
        vehicle = f"VH-{random.randint(1, 5):03d}"
    else:
        vehicle = f"VH-{random.randint(6, 200):03d}"
    
    ts = BASE_DATE + timedelta(days=random.randint(0, 364), hours=random.randint(0, 23),
                                minutes=random.randint(0, 59), seconds=random.randint(0, 59))
    telemetry.append((
        str(uuid.uuid4()),                          # reading_id (telematics gateway message id)
        vehicle, ts,
        __builtins__.round(random.uniform(0, 120), 1),       # speed_kmh
        __builtins__.round(random.uniform(0, 100), 1),        # fuel_pct
        __builtins__.round(random.gauss(22, 8), 1),           # engine_temp_c
        __builtins__.round(random.gauss(4, 3), 1) if random.random() < 0.3 else None,  # cargo_temp (only cold chain)
        random.randint(0, 500000),               # odometer_km
        random.choice(["idle", "moving", "loading", "maintenance"]),
    ))

tel_schema = StructType([
    StructField("reading_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("reading_timestamp", TimestampType()),
    StructField("speed_kmh", DoubleType()), StructField("fuel_pct", DoubleType()),
    StructField("engine_temp_c", DoubleType()), StructField("cargo_temp_c", DoubleType(), True),
    StructField("odometer_km", IntegerType()), StructField("vehicle_status", StringType()),
])

df_telemetry = spark.createDataFrame(telemetry, tel_schema)
(df_telemetry.repartition(40)  # ~5K/file, ~1 MB
   .write.mode("overwrite")  # initial seed only
   .json(f"{LANDING_BASE}/telemetry"))

In [0]:
df_telemetry_validate = spark.read.json(f"{LANDING_BASE}/telemetry/")
print(f"✓ vehicle_telemetry: {df_telemetry_validate.count()} records (intentionally skewed — 60% from 5 vehicles)")


In [ ]:
# ── VEHICLES (fleet master, 200 vehicles) — one-time JSON snapshot, SCD1 source ──
# The fleet master behind VH-001..VH-200 (same ids as telemetry / shipment events).
# Delivered as JSON on the landing volume like the other file feeds: this seed is the
# first drop; the incremental generator later appends rare FULL-ROW post-images
# (depot reassignments), so downstream stays a plain SCD1 upsert. Deliberately
# simpler than orders: no sparse updates, no snapshot-vs-increment semantics.
vehicle_models = {
    "pony_cart":  ["Shire Cob Cart", "Bree Trotter", "Buckland Runner", "Michel Delving Dray"],
    "wagon":      ["Rohan Wain XL", "Gondor Grain Wain", "Erebor Ore Hauler", "Esgaroth Freight Wain"],
    "cold_wagon": ["Forochel Ice Wain", "Misty Mountain Cooler", "Helcaraxë Reefer"],
}
depot_cities = ["Minas Tirith", "Edoras", "Hobbiton", "Bree", "Pelargir", "Dol Amroth", "Esgaroth", "Rivendell"]
plate_realm = {"Minas Tirith": "GO", "Edoras": "RO", "Hobbiton": "SH", "Bree": "BR",
               "Pelargir": "GO", "Dol Amroth": "GO", "Esgaroth": "DA", "Rivendell": "RV"}
capacity_by_type = {"pony_cart": [1200, 1500], "wagon": [8000, 12000, 18000], "cold_wagon": [6000, 10000]}

vehicle_records = []
for vid in range(1, 201):
    # ~30% cold-chain capable (cold wagons), consistent with ~30% of telemetry readings carrying cargo_temp_c
    vtype = random.choices(["pony_cart", "wagon", "cold_wagon"], weights=[45, 25, 30])[0]
    depot = random.choice(depot_cities)
    vehicle_records.append((
        f"VH-{vid:03d}",
        f"{plate_realm[depot]}-{random.randint(10000, 999999)}",
        random.choice(vehicle_models[vtype]),
        vtype,
        random.choice(capacity_by_type[vtype]),
        vtype == "cold_wagon",
        depot,
        BASE_DATE - timedelta(days=random.randint(90, 3000)),  # commissioned_date
        BASE_DATE,                                             # last_updated as of the seed
    ))

veh_schema = StructType([
    StructField("vehicle_id", StringType()), StructField("plate_number", StringType()),
    StructField("model", StringType()), StructField("vehicle_type", StringType()),
    StructField("capacity_kg", IntegerType()), StructField("cold_chain_capable", BooleanType()),
    StructField("home_depot", StringType()),
    StructField("commissioned_date", TimestampType()), StructField("last_updated", TimestampType()),
])

df_vehicles = spark.createDataFrame(vehicle_records, veh_schema)
(df_vehicles.repartition(1)   # 200 rows, one small file
   .write.mode("overwrite")   # initial seed only
   .json(f"{LANDING_BASE}/vehicles"))


In [ ]:
df_vehicles_validate = spark.read.json(f"{LANDING_BASE}/vehicles/")
n_cold = df_vehicles_validate.filter("cold_chain_capable").count()
print(f"✓ vehicles: {df_vehicles_validate.count()} fleet-master rows ({n_cold} cold-chain capable)")


In [ ]:
# ── SUMMARY ──
print("=" * 60)
print("Shadowfax Logistics Data Platform — Source Data Ready")
print("=" * 60)
# Delta source (current-state, CDF on): customers
cust_n = spark.table(f"{SCHEMA_INGEST}.customers").count()
print(f"  {SCHEMA_INGEST}.customers (Delta, CDF): {cust_n:,} rows")
# File-CDC / JSON landing feeds: orders (snapshot), shipment_events, telemetry
for feed in ["orders", "shipment_events", "telemetry", "vehicles"]:
    files = [f for f in dbutils.fs.ls(f"{LANDING_BASE}/{feed}/") if f.name.lower().endswith(".json")]
    rows = spark.read.json(f"{LANDING_BASE}/{feed}/").count()
    print(f"  {LANDING_BASE}/{feed}: {rows:,} rows across {len(files)} files")
print()
print("Data characteristics:")
print("  • customers: current-state source (Postgres-style, CDF on); feeds the SCD2 dimension")
print("  • orders: file-based CDC; SNAPSHOT of 50K orders, ~2% NULL amounts; INSERT/UPDATE drops arrive via the generator")
print("  • shipment_events: ~3% duplicates (dedup practice)")
print("  • vehicle_telemetry: 60% skewed to 5 vehicles (skew handling)")
print("  • vehicles: fleet master (SCD1); rare depot reassignments arrive via the generator")
print()
print("Next: run the bronze ingestion")
